In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark=SparkSession.builder.appName("Capstonepyspark2").getOrCreate()

In [3]:
%%writefile customers.csv
customer_id,customer_name,city,state,age,gender,plan_id,status
101,Rahul Sharma,Hyderabad,Telangana,35,Male,P101,Active
102,Priya Reddy,Bangalore,Karnataka,29,Female,P102,Active
103,Amit Kumar,Mumbai,Maharashtra,42,Male,P103,Inactive
104,Sneha Patel,Chennai,Tamil Nadu,31,Female,P101,Active
105,Farhan Ali,Delhi,Delhi,55,Male,P104,Active
106,Neha Singh,Pune,Maharashtra,38,Female,P102,Active
107,Arjun Verma,Hyderabad,Telangana,26,Male,P103,Inactive
108,Meera Nair,Kochi,Kerala,48,Female,P104,Active
109,Kiran Rao,Bangalore,Karnataka,33,Male,P101,Active
110,Nisha Reddy,Delhi,Delhi,41,Female,P102,Active
111,Ravi Kumar,Mumbai,Maharashtra,45,Male,P105,Active
112,Ayesha Khan,Hyderabad,Telangana,28,Female,,Active

Writing customers.csv


In [4]:
%%writefile usage.csv
usage_id,customer_id,usage_month,data_used_gb,call_minutes,sms_count
1001,101,2026-01,45,900,120
1002,102,2026-01,30,600,80
1003,103,2026-01,12,250,40
1004,104,2026-01,55,1100,150
1005,105,2026-01,75,1500,200
1006,106,2026-01,28,500,60
1007,107,2026-01,10,200,20
1008,108,2026-01,80,1600,250
1009,109,2026-01,48,950,100
1010,110,2026-01,32,700,90
1011,120,2026-01,60,1300,140
1012,101,2026-02,50,1000,130
1013,102,2026-02,34,650,85
1014,104,2026-02,58,1200,160
1015,105,2026-02,,1450,210


Writing usage.csv


In [5]:
%%writefile plans.json
[
 {
 "plan_id": "P101",
 "plan_name": "Smart Basic",
 "monthly_fee": 499,
 "data_limit_gb": 50,
 "features": {
 "unlimited_calls": true,
 "ott_included": false,
 "roaming": "National"
 }
 },
 {
 "plan_id": "P102",
 "plan_name": "Smart Plus",
 "monthly_fee": 799,
 "data_limit_gb": 75,
 "features": {
 "unlimited_calls": true,
 "ott_included": true,
 "roaming": "National"
 }
 },
 {
 "plan_id": "P103",
 "plan_name": "Budget Saver",
 "monthly_fee": 299,
 "data_limit_gb": 25,
 "features": {
 "unlimited_calls": false,
 "ott_included": false,
 "roaming": null
 }
 },
 {
 "plan_id": "P104",
 "plan_name": "Premium Max",
 "monthly_fee": 1199,
 "data_limit_gb": 100,
 "features": {
 "unlimited_calls": true,
 "ott_included": true,
 "roaming": "International"
 }
 }
]


Writing plans.json


In [6]:
%%writefile payments.csv
payment_id,customer_id,bill_month,amount_paid,payment_mode,payment_status
5001,101,2026-01,499,UPI,Success
5002,102,2026-01,799,Card,Success
5003,103,2026-01,299,Cash,Failed
5004,104,2026-01,499,UPI,Success
5005,105,2026-01,1199,Card,Success
5006,106,2026-01,799,UPI,Success
5007,107,2026-01,299,Cash,Pending
5008,108,2026-01,1199,Card,Success
5009,109,2026-01,499,UPI,Success
5010,110,2026-01,799,UPI,Success
5011,112,2026-01,,UPI,Success
5012,101,2026-02,499,Card,Success
5013,102,2026-02,799,UPI,Success
5014,104,2026-02,499,UPI,Success
5015,105,2026-02,1199,,Pending

Writing payments.csv


In [7]:
customersdf=spark.read.csv("customers.csv",header=True,inferSchema=True)

In [8]:
usagedf=spark.read.csv("usage.csv",header=True,inferSchema=True)

In [9]:
paymentsdf=spark.read.csv("payments.csv",header=True,inferSchema=True)

In [10]:
plansdf=spark.read.option("multiline","true").json("plans.json")

In [11]:
customersdf.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- status: string (nullable = true)



In [12]:
usagedf.printSchema()

root
 |-- usage_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- usage_month: timestamp (nullable = true)
 |-- data_used_gb: integer (nullable = true)
 |-- call_minutes: integer (nullable = true)
 |-- sms_count: integer (nullable = true)



In [13]:
paymentsdf.printSchema()

root
 |-- payment_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- bill_month: timestamp (nullable = true)
 |-- amount_paid: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)



In [14]:
plansdf.printSchema()

root
 |-- data_limit_gb: long (nullable = true)
 |-- features: struct (nullable = true)
 |    |-- ott_included: boolean (nullable = true)
 |    |-- roaming: string (nullable = true)
 |    |-- unlimited_calls: boolean (nullable = true)
 |-- monthly_fee: long (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- plan_name: string (nullable = true)



In [15]:
print(customersdf.count())

12


In [16]:
print(usagedf.count())

15


In [17]:
print(paymentsdf.count())

15


In [18]:
print(plansdf.count())

4


In [19]:
customersdf.write.mode("overwrite").parquet("bronze/customers")

In [20]:
usagedf.write.mode("overwrite").parquet("bronze/usage")

In [21]:
paymentsdf.write.mode("overwrite").parquet("bronze/payments")

In [22]:
plansdf.write.mode("overwrite").parquet("bronze/plans")

In [23]:
customersdf.filter(col("plan_id").isNull()).show()

+-----------+-------------+---------+---------+---+------+-------+------+
|customer_id|customer_name|     city|    state|age|gender|plan_id|status|
+-----------+-------------+---------+---------+---+------+-------+------+
|        112|  Ayesha Khan|Hyderabad|Telangana| 28|Female|   NULL|Active|
+-----------+-------------+---------+---------+---+------+-------+------+



In [24]:
usagedf.filter(col("data_used_gb").isNull()).show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1015|        105|2026-02-01 00:00:00|        NULL|        1450|      210|
+--------+-----------+-------------------+------------+------------+---------+



In [25]:
paymentsdf.filter(col("amount_paid").isNull()).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5011|        112|2026-01-01 00:00:00|       NULL|         UPI|       Success|
+----------+-----------+-------------------+-----------+------------+--------------+



In [26]:
paymentsdf.filter(col("payment_mode").isNull()).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5015|        105|2026-02-01 00:00:00|       1199|        NULL|       Pending|
+----------+-----------+-------------------+-----------+------------+--------------+



In [27]:
usagedf=usagedf.fillna({"data_used_gb":0})

In [28]:
paymentsdf=paymentsdf.fillna({"amount_paid":0,"payment_mode":"Not Provided"})

In [29]:
customersdf=customersdf.fillna({"plan_id":"UNKNOWN"})

In [30]:
customersdf=customersdf.withColumn("data_quality_status",when(col("plan_id")=="UNKNOWN","Issue").otherwise("Valid"))

In [31]:
usagedf=usagedf.withColumn("data_quality_status",when(col("data_used_gb")==0,"Issue").otherwise("Valid"))

In [32]:
paymentsdf=paymentsdf.withColumn("data_quality_status",when((col("amount_paid")==0)|(col("payment_mode")=="Not Provided"),"Issue").otherwise("Valid"))

In [33]:
customersdf.write.mode("overwrite").parquet("silver/customers")

In [34]:
usagedf.write.mode("overwrite").parquet("silver/usage")

In [35]:
paymentsdf.write.mode("overwrite").parquet("silver/payments")

In [36]:
plansflatdf=plansdf.select("plan_id","plan_name","monthly_fee","data_limit_gb",col("features.unlimited_calls").alias("unlimited_calls"),col("features.ott_included").alias("ott_included"),col("features.roaming").alias("roaming"))

In [37]:
plansflatdf=plansflatdf.fillna({"roaming":"Not Available"})

In [38]:
plansflatdf.write.mode("overwrite").parquet("silver/plans")

In [39]:
customerplandf=customersdf.join(plansflatdf,"plan_id","left")

In [40]:
customerusagedf=customersdf.join(usagedf,"customer_id","left")

In [41]:
customerpaymentdf=customersdf.join(paymentsdf,"customer_id","left")

In [42]:
finaldf=customersdf.join(plansflatdf,"plan_id","left") \
    .join(usagedf,"customer_id","left") \
    .join(paymentsdf,"customer_id","left")

In [43]:
customersdf.join(plansflatdf,"plan_id","left_anti").show()

+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|status|data_quality_status|
+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+
|   P105|        111|   Ravi Kumar|   Mumbai|Maharashtra| 45|  Male|Active|              Valid|
|UNKNOWN|        112|  Ayesha Khan|Hyderabad|  Telangana| 28|Female|Active|              Issue|
+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+



In [44]:
usagedf.join(customersdf,"customer_id","left_anti").show()

+-----------+--------+-------------------+------------+------------+---------+-------------------+
|customer_id|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+-----------+--------+-------------------+------------+------------+---------+-------------------+
|        120|    1011|2026-01-01 00:00:00|          60|        1300|      140|              Valid|
+-----------+--------+-------------------+------------+------------+---------+-------------------+



In [45]:
paymentsdf.join(customersdf,"customer_id","left_anti").show()

+-----------+----------+----------+-----------+------------+--------------+-------------------+
|customer_id|payment_id|bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+----------+----------+-----------+------------+--------------+-------------------+
+-----------+----------+----------+-----------+------------+--------------+-------------------+



In [46]:
finaldf=finaldf.withColumn(
    "usage_category",
    when(col("data_used_gb")>=70,"Heavy User")
    .when(col("data_used_gb")>=30,"Medium User")
    .otherwise("Low User")
)

In [47]:
finaldf=finaldf.withColumn(
    "payment_category",
    when(col("amount_paid")>=1000,"High Payment")
    .when(col("amount_paid")>=500,"Medium Payment")
    .otherwise("Low Payment")
)

In [48]:
finaldf=finaldf.withColumn(
    "churn_risk",
    when(
        (col("status")=="Inactive")|\
        (col("payment_status")!="Success"),
        "High Risk"
    )
    .when(col("data_used_gb")<15,"Medium Risk")
    .otherwise("Low Risk")
)

In [49]:
finaldf=finaldf.withColumn(
    "over_usage_gb",
    col("data_used_gb")-col("data_limit_gb")
)

In [50]:
finaldf=finaldf.withColumn(
    "over_usage_flag",
    when(col("over_usage_gb")>0,"Yes")
    .otherwise("No")
)

In [51]:
customersdf.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    3|
+---------+-----+



In [52]:
customersdf.groupBy("state").count().show()

+-----------+-----+
|      state|count|
+-----------+-----+
|  Karnataka|    2|
|     Kerala|    1|
| Tamil Nadu|    1|
|      Delhi|    2|
|  Telangana|    3|
|Maharashtra|    3|
+-----------+-----+



In [53]:
customersdf.groupBy("plan_id").count().show()

+-------+-----+
|plan_id|count|
+-------+-----+
|   P105|    1|
|   P102|    3|
|UNKNOWN|    1|
|   P103|    2|
|   P104|    2|
|   P101|    3|
+-------+-----+



In [54]:
finaldf.groupBy("usage_category").count().show()

+--------------+-----+
|usage_category|count|
+--------------+-----+
|   Medium User|   14|
|    Heavy User|    3|
|      Low User|    7|
+--------------+-----+



In [55]:
finaldf.groupBy("churn_risk").count().show()

+-----------+-----+
| churn_risk|count|
+-----------+-----+
|   Low Risk|   19|
|Medium Risk|    1|
|  High Risk|    4|
+-----------+-----+



In [56]:
finaldf.groupBy("plan_name").agg(sum("data_used_gb").alias("total_data_usage")).show()

+------------+----------------+
|   plan_name|total_data_usage|
+------------+----------------+
|        NULL|            NULL|
| Smart Basic|             464|
|Budget Saver|              22|
| Premium Max|             230|
|  Smart Plus|             188|
+------------+----------------+



In [57]:
finaldf.groupBy("plan_name").agg(avg("data_used_gb").alias("average_data_usage")).show()

+------------+------------------+
|   plan_name|average_data_usage|
+------------+------------------+
|        NULL|              NULL|
| Smart Basic| 51.55555555555556|
|Budget Saver|              11.0|
| Premium Max|              46.0|
|  Smart Plus|31.333333333333332|
+------------+------------------+



In [58]:
finaldf.groupBy("city").agg(sum("call_minutes").alias("total_call_minutes")).show()

+---------+------------------+
|     city|total_call_minutes|
+---------+------------------+
|Bangalore|              3450|
|    Kochi|              1600|
|  Chennai|              4600|
|   Mumbai|               250|
|     Pune|               500|
|    Delhi|              6600|
|Hyderabad|              4000|
+---------+------------------+



In [59]:
finaldf.groupBy("state").agg(sum("sms_count").alias("total_sms_count")).show()

+-----------+---------------+
|      state|total_sms_count|
+-----------+---------------+
|  Karnataka|            430|
|     Kerala|            250|
| Tamil Nadu|            620|
|      Delhi|            910|
|  Telangana|            520|
|Maharashtra|            100|
+-----------+---------------+



In [60]:
finaldf.filter(col("payment_status")=="Success").agg(sum("amount_paid").alias("total_successful_revenue")).show()

+------------------------+
|total_successful_revenue|
+------------------------+
|                   12882|
+------------------------+



In [61]:
finaldf.groupBy("city").agg(sum("amount_paid").alias("total_revenue")).show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|         3695|
|    Kochi|         1199|
|  Chennai|         1996|
|   Mumbai|          299|
|     Pune|          799|
|    Delhi|         5595|
|Hyderabad|         2295|
+---------+-------------+



In [62]:
finaldf.groupBy("plan_name").agg(sum("amount_paid").alias("total_revenue")).show()

+------------+-------------+
|   plan_name|total_revenue|
+------------+-------------+
|        NULL|            0|
| Smart Basic|         4491|
|Budget Saver|          598|
| Premium Max|         5995|
|  Smart Plus|         4794|
+------------+-------------+



In [63]:
finaldf.groupBy("payment_mode").agg(sum("amount_paid").alias("total_revenue")).show()

+------------+-------------+
|payment_mode|total_revenue|
+------------+-------------+
|        NULL|         NULL|
|        Card|         6193|
|        Cash|          598|
|Not Provided|         2398|
|         UPI|         6689|
+------------+-------------+



In [64]:
finaldf.groupBy("plan_name").agg(sum("amount_paid").alias("total_revenue")).orderBy(desc("total_revenue")).show(1)

+-----------+-------------+
|  plan_name|total_revenue|
+-----------+-------------+
|Premium Max|         5995|
+-----------+-------------+
only showing top 1 row


In [65]:
finaldf.groupBy("city").agg(sum("amount_paid").alias("total_revenue")).orderBy(desc("total_revenue")).show(1)

+-----+-------------+
| city|total_revenue|
+-----+-------------+
|Delhi|         5595|
+-----+-------------+
only showing top 1 row


In [66]:
usagewindow=Window.orderBy(desc("data_used_gb"))

In [67]:
paymentwindow=Window.orderBy(desc("amount_paid"))

In [68]:
finaldf.withColumn("usage_rank",rank().over(usagewindow)).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+----------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|usage_rank|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+----------

In [69]:
finaldf.withColumn("payment_rank",rank().over(paymentwindow)).show()

+-----------+-------+-------------+---------+-----------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+---------------+------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|payment_rank|
+-----------+-------+-------------+---------+-----------+---+------+------+-------------------+-----------+-----------+-

In [70]:
finaldf.orderBy(desc("data_used_gb")).show(3)

+-----------+-------+-------------+-----+------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+
|customer_id|plan_id|customer_name| city| state|age|gender|status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|
+-----------+-------+-------------+-----+------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------

In [71]:
finaldf.orderBy(desc("amount_paid")).show(3)

+-----------+-------+-------------+-----+-----+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+---------------+
|customer_id|plan_id|customer_name| city|state|age|gender|status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|
+-----------+-------+-------------+-----+-----+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------+

In [72]:
citywindow=Window.partitionBy("city").orderBy(desc("data_used_gb"))

In [73]:
finaldf.withColumn("row_num",row_number().over(citywindow)).filter(col("row_num")==1).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+-------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|row_num|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+----

In [74]:
planwindow=Window.partitionBy("plan_name").orderBy(desc("data_used_gb"))

In [75]:
finaldf.withColumn("row_num",row_number().over(planwindow)).filter(col("row_num")==1).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+-------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|row_num|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+----

In [76]:
revenuewindow=Window.orderBy("bill_month")

In [77]:
finaldf.withColumn("running_total_revenue",sum("amount_paid").over(revenuewindow)).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+---------------+---------------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|running_total_revenue|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------

In [78]:
usagemonthwindow=Window.partitionBy("customer_id").orderBy("usage_month")

In [79]:
finaldf.withColumn("previous_month_usage",lag("data_used_gb").over(usagemonthwindow)).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+---------------+--------------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|previous_month_usage|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+-

In [80]:
finaldf.withColumn("next_month_usage",lead("data_used_gb").over(usagemonthwindow)).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+---------------+----------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|next_month_usage|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+---------

In [81]:
finaldf.withColumn("previous_month_usage",lag("data_used_gb").over(usagemonthwindow)).filter(col("data_used_gb")>col("previous_month_usage")).show()

+-----------+-------+-------------+---------+----------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+--------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+--------------------+
|customer_id|plan_id|customer_name|     city|     state|age|gender|status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included| roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|previous_month_usage|
+-----------+-------+-------------+---------+----------+---+------+------+-------------------+-----------+-----------+

In [82]:
customersdf.createOrReplaceTempView("customers")

In [83]:
usagedf.createOrReplaceTempView("usage")

In [84]:
paymentsdf.createOrReplaceTempView("payments")

In [85]:
plansflatdf.createOrReplaceTempView("plans")

In [86]:
finaldf.createOrReplaceTempView("telecom")

In [87]:
spark.sql("SELECT * FROM telecom WHERE status='Active'").show()

+-----------+-------+-------------+---------+-----------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+---------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|
+-----------+-------+-------------+---------+-----------+---+------+------+-------------------+-----------+-----------+-------------+-------------

In [88]:
spark.sql("SELECT city,COUNT(*) AS total_customers FROM telecom GROUP BY city").show()

+---------+---------------+
|     city|total_customers|
+---------+---------------+
|Bangalore|              5|
|    Kochi|              1|
|  Chennai|              4|
|   Mumbai|              2|
|     Pune|              1|
|    Delhi|              5|
|Hyderabad|              6|
+---------+---------------+



In [89]:
spark.sql("SELECT plan_name,SUM(amount_paid) AS total_revenue FROM telecom GROUP BY plan_name").show()

+------------+-------------+
|   plan_name|total_revenue|
+------------+-------------+
|        NULL|            0|
| Smart Basic|         4491|
|Budget Saver|          598|
| Premium Max|         5995|
|  Smart Plus|         4794|
+------------+-------------+



In [90]:
spark.sql("SELECT * FROM telecom WHERE usage_category='Heavy User'").show()

+-----------+-------+-------------+-----+------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+
|customer_id|plan_id|customer_name| city| state|age|gender|status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|
+-----------+-------+-------------+-----+------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------

In [91]:
spark.sql("SELECT * FROM telecom WHERE churn_risk='High Risk'").show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+------

In [92]:
spark.sql("SELECT * FROM telecom WHERE plan_id='UNKNOWN' OR plan_name IS NULL").show()

+-----------+-------+-------------+---------+-----------+---+------+------+-------------------+---------+-----------+-------------+---------------+------------+-------+--------+-----------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|status|data_quality_status|plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming|usage_id|usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|
+-----------+-------+-------------+---------+-----------+---+------+------+-------------------+---------+-----------+-------------+---------------+------------+-------+--------+---

In [93]:
spark.sql("SELECT * FROM telecom WHERE payment_status IN ('Failed','Pending')").show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+------

In [94]:
spark.sql("SELECT customer_id,customer_name,data_used_gb FROM telecom ORDER BY data_used_gb DESC LIMIT 5").show()

+-----------+-------------+------------+
|customer_id|customer_name|data_used_gb|
+-----------+-------------+------------+
|        108|   Meera Nair|          80|
|        105|   Farhan Ali|          75|
|        105|   Farhan Ali|          75|
|        104|  Sneha Patel|          58|
|        104|  Sneha Patel|          58|
+-----------+-------------+------------+



In [95]:
spark.sql("SELECT payment_mode,SUM(amount_paid) AS total_revenue FROM telecom GROUP BY payment_mode").show()

+------------+-------------+
|payment_mode|total_revenue|
+------------+-------------+
|        NULL|         NULL|
|        Card|         6193|
|        Cash|          598|
|Not Provided|         2398|
|         UPI|         6689|
+------------+-------------+



In [98]:
finaldf.write.mode("overwrite").partitionBy("usage_month").parquet("gold/telecomanalytics")

In [100]:
%%writefile usage_2026_03.csv
usage_id,customer_id,usage_month,data_used_gb,call_minutes,sms_count
1016,101,2026-03,52,1050,135
1017,102,2026-03,38,720,92
1018,104,2026-03,62,1250,170
1019,105,2026-03,78,1520,220
1020,108,2026-03,85,1680,260
1021,109,2026-03,50,980,110
1022,110,2026-03,35,760,95
1023,112,2026-03,25,450,55

Writing usage_2026_03.csv


In [101]:
incrementalusagedf=spark.read.csv("usage_2026_03.csv",header=True,inferSchema=True)

In [102]:
incrementalusagedf=spark.read.csv("usage_2026_03.csv",header=True,inferSchema=True)

In [103]:
beforecount=spark.read.parquet("silver/usage").count()

In [104]:
incrementalusagedf.write.mode("append").parquet("silver/usage")

In [105]:
usagedf=spark.read.parquet("silver/usage")

In [106]:
finaldf=customersdf.join(plansflatdf,"plan_id","left").join(usagedf,"customer_id","left").join(paymentsdf,"customer_id","left")

In [113]:
finaldf=customersdf.drop("data_quality_status") \
    .join(plansflatdf,"plan_id","left") \
    .join(usagedf.drop("data_quality_status"),"customer_id","left") \
    .join(paymentsdf.drop("data_quality_status"),"customer_id","left")

In [114]:
finaldf=finaldf.withColumn("usage_category",when(col("data_used_gb")>=70,"Heavy User").when(col("data_used_gb")>=30,"Medium User").otherwise("Low User"))

In [115]:
finaldf=finaldf.withColumn("payment_category",when(col("amount_paid")>=1000,"High Payment").when(col("amount_paid")>=500,"Medium Payment").otherwise("Low Payment"))

In [116]:
finaldf=finaldf.withColumn("churn_risk",when((col("status")=="Inactive")|(col("payment_status")!="Success"),"High Risk").when(col("data_used_gb")<15,"Medium Risk").otherwise("Low Risk"))

In [117]:
finaldf=finaldf.withColumn("over_usage_gb",col("data_used_gb")-col("data_limit_gb"))

In [118]:
finaldf=finaldf.withColumn("over_usage_flag",when(col("over_usage_gb")>0,"Yes").otherwise("No"))

In [119]:
finaldf.write.mode("overwrite").partitionBy("usage_month").parquet("gold/telecomanalytics")

In [120]:
aftercount=usagedf.count()

In [121]:
print(beforecount)

15


In [122]:
print(aftercount)

23


In [123]:
customerusagesummarydf=finaldf.select("customer_id","customer_name","city","plan_name","usage_month","data_used_gb","data_limit_gb","over_usage_flag","amount_paid","payment_status","churn_risk")

In [124]:
customerusagesummarydf.show()

+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+----------+
|customer_id|customer_name|     city|   plan_name|        usage_month|data_used_gb|data_limit_gb|over_usage_flag|amount_paid|payment_status|churn_risk|
+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+----------+
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-03-01 00:00:00|          52|           50|            Yes|        499|       Success|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-03-01 00:00:00|          52|           50|            Yes|        499|       Success|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-02-01 00:00:00|          50|           50|             No|        499|       Success|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-02-01 00:00:00|          50|     

In [126]:
from pyspark.sql.functions import countDistinct

In [127]:
planperformancedf=finaldf.groupBy("plan_name").agg(countDistinct("customer_id").alias("total_customers"),sum("data_used_gb").alias("total_data_usage"),avg("data_used_gb").alias("average_data_usage"),sum("amount_paid").alias("total_revenue"))

In [128]:
planperformancedf.show()

+------------+---------------+----------------+------------------+-------------+
|   plan_name|total_customers|total_data_usage|average_data_usage|total_revenue|
+------------+---------------+----------------+------------------+-------------+
|        NULL|              2|              25|              25.0|            0|
| Smart Basic|              3|             742|              53.0|         6986|
|Budget Saver|              2|              22|              11.0|          598|
| Premium Max|              2|             471|            58.875|         9592|
|  Smart Plus|              3|             299| 33.22222222222222|         7191|
+------------+---------------+----------------+------------------+-------------+



In [129]:
cityrevenuedf=finaldf.groupBy("city").agg(countDistinct("customer_id").alias("total_customers"),sum("amount_paid").alias("total_revenue"),avg("amount_paid").alias("average_payment"))

In [130]:
cityrevenuedf.show()

+---------+---------------+-------------+---------------+
|     city|total_customers|total_revenue|average_payment|
+---------+---------------+-------------+---------------+
|Bangalore|              2|         5792|          724.0|
|    Kochi|              1|         2398|         1199.0|
|  Chennai|              1|         2994|          499.0|
|   Mumbai|              2|          299|          299.0|
|     Pune|              1|          799|          799.0|
|    Delhi|              2|         8792|         1099.0|
|Hyderabad|              3|         3293|        411.625|
+---------+---------------+-------------+---------------+



In [131]:
churnriskdf=finaldf.select("customer_id","customer_name","city","plan_name","payment_status","status","churn_risk")

In [132]:
churnriskdf.show()

+-----------+-------------+---------+------------+--------------+--------+----------+
|customer_id|customer_name|     city|   plan_name|payment_status|  status|churn_risk|
+-----------+-------------+---------+------------+--------------+--------+----------+
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|  Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|  Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|  Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|    

In [133]:
overusagedf=finaldf.select("customer_id","customer_name","plan_name","data_used_gb","data_limit_gb","over_usage_gb")

In [134]:
overusagedf.show()

+-----------+-------------+------------+------------+-------------+-------------+
|customer_id|customer_name|   plan_name|data_used_gb|data_limit_gb|over_usage_gb|
+-----------+-------------+------------+------------+-------------+-------------+
|        101| Rahul Sharma| Smart Basic|          52|           50|            2|
|        101| Rahul Sharma| Smart Basic|          52|           50|            2|
|        101| Rahul Sharma| Smart Basic|          50|           50|            0|
|        101| Rahul Sharma| Smart Basic|          50|           50|            0|
|        101| Rahul Sharma| Smart Basic|          45|           50|           -5|
|        101| Rahul Sharma| Smart Basic|          45|           50|           -5|
|        102|  Priya Reddy|  Smart Plus|          38|           75|          -37|
|        102|  Priya Reddy|  Smart Plus|          38|           75|          -37|
|        102|  Priya Reddy|  Smart Plus|          34|           75|          -41|
|        102|  P